# Salesforce Synthetic Data Generator

This notebook dynamically generates realistic synthetic Salesforce **Account**, **Opportunity**, and **Case** records for testing, development, and demo purposes, then inserts them directly into Salesforce via the REST API.

All records are relationally linked — Opportunities and Cases reference valid Account IDs. Data can also be exported locally as CSV, JSON, or a multi-sheet Excel workbook.

## 1. Import Required Libraries

In [3]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for _pkg in ["faker", "pandas", "numpy", "openpyxl", "simple-salesforce", "python-dotenv"]:
    _import_name = {
        "simple-salesforce": "simple_salesforce",
        "python-dotenv":     "dotenv",
    }.get(_pkg, _pkg)
    try:
        __import__(_import_name)
    except ImportError:
        print(f"Installing {_pkg}...")
        _install(_pkg)

import os, random, string
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from faker import Faker
from simple_salesforce import Salesforce, SalesforceLogin, SFType

# ─────────────────────────────────────────────────────────────────────────────
# .env file location
# Defaults to auto-discovery (walks up from the notebook directory to find .env
# in the project root). Override with an explicit path if needed, e.g.:
#   DOTENV_PATH = r"C:\Users\you\projects\my-project\.env"
# ─────────────────────────────────────────────────────────────────────────────
DOTENV_PATH = find_dotenv()  # set to an explicit path string to override

# override=True ensures .env always wins over stale kernel environment variables
load_dotenv(DOTENV_PATH, override=True)
print(f".env loaded from: {DOTENV_PATH or '(not found)'}")

fake = Faker("en_US")
Faker.seed(42)
random.seed(42)
np.random.seed(42)

print("All libraries imported successfully.")


Installing faker...
Installing pandas...
Installing openpyxl...
Installing simple-salesforce...
Installing python-dotenv...
.env loaded from: c:\Users\JimmyBilly\VSCode\MW-Toolbox\.env
All libraries imported successfully.


## 2. Define Configuration Parameters

Adjust these values to control the volume and characteristics of the generated data.

### .env Configuration

Edit the existing `.env` file in the project root, or create one if it doesn't already exist (it is already `.gitignore`d). Add the following keys to the Salesforce section — data generation parameters are hard-coded in the notebook and do not need to be set here.

- **`SF_AUTH_METHOD`** — `client_credentials` (Option B) or `password` (Option A)
- **`SF_MY_DOMAIN`** — Your org's My Domain hostname, e.g. `orgfarm-b24e82440c-dev-ed.develop.my.salesforce.com` (used as the OAuth token endpoint for client credentials flow; find it under Setup → My Domain)
- **`SF_DOMAIN`** — `login` for production, `test` for sandbox (used only for password auth)
- **`SF_API_BATCH_SIZE`** — Max records per REST API request (200 is the Salesforce limit)
- **`SF_OUTPUT_DIR`** — Local output directory for CSV / JSON / Excel exports
- **`SF_CLIENT_ID`** / **`SF_CLIENT_SECRET`** — Connected App Consumer Key / Secret (Option B)
- **`SF_USERNAME`** / **`SF_PASSWORD`** / **`SF_SECURITY_TOKEN`** — Username + password auth (Option A)


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Record volume — edit these directly to control how much data is generated
# ─────────────────────────────────────────────────────────────────────────────
NUM_ACCOUNTS          = 50
MIN_OPPS_PER_ACCOUNT  = 1
MAX_OPPS_PER_ACCOUNT  = 5
MIN_CASES_PER_ACCOUNT = 0
MAX_CASES_PER_ACCOUNT = 8

# ─────────────────────────────────────────────────────────────────────────────
# Date ranges
# ─────────────────────────────────────────────────────────────────────────────
CREATED_DATE_START      = datetime(2022, 1, 1)
CREATED_DATE_END        = datetime(2025, 12, 31)
CLOSE_DATE_FUTURE_DAYS  = 365
PIPELINE_LOOKBACK_DAYS  = 180

# Existing opportunity lifecycle simulation settings
PAST_OPP_UPDATE_LIMIT   = 200
PAST_OPP_BASE_WIN_RATE  = 0.57

# Oil & Gas services conversion profile tuning
OIL_GAS_STAGE_WIN_RATE = {
    "Prospecting": 0.08,
    "Qualification": 0.14,
    "Needs Analysis": 0.22,
    "Value Proposition": 0.32,
    "Id. Decision Makers": 0.42,
    "Perception Analysis": 0.50,
    "Proposal/Price Quote": 0.58,
    "Negotiation/Review": 0.68,
}

OIL_GAS_HORIZON_STAGE_WEIGHTS = {
    "near": {
        "stages": ["Negotiation/Review", "Proposal/Price Quote", "Perception Analysis", "Id. Decision Makers"],
        "weights": [0.45, 0.30, 0.15, 0.10],
    },
    "mid": {
        "stages": ["Proposal/Price Quote", "Perception Analysis", "Id. Decision Makers", "Value Proposition", "Negotiation/Review"],
        "weights": [0.30, 0.25, 0.20, 0.15, 0.10],
    },
    "far": {
        "stages": ["Qualification", "Needs Analysis", "Value Proposition", "Prospecting", "Id. Decision Makers"],
        "weights": [0.28, 0.27, 0.20, 0.15, 0.10],
    },
    "long": {
        "stages": ["Prospecting", "Qualification", "Needs Analysis", "Value Proposition"],
        "weights": [0.40, 0.30, 0.20, 0.10],
    },
}

# ─────────────────────────────────────────────────────────────────────────────
# Local output directory for CSV / JSON / Excel exports
# (override via SF_OUTPUT_DIR in .env)
# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = os.getenv("SF_OUTPUT_DIR", "c:/temp")

# ─────────────────────────────────────────────────────────────────────────────
# Domain: Industrial Equipment Seller — Energy Sector Customers
# ─────────────────────────────────────────────────────────────────────────────

ENERGY_COMPANY_PREFIXES = [
    "Permian", "Gulf Coast", "Lone Star", "Eagle Ford", "Bakken", "Marcellus",
    "Rocky Mountain", "Appalachian", "Midcontinent", "Trans-Western", "Southern Plains",
    "Continental", "Pacific Rim", "Anadarko", "Delaware Basin", "Haynesville",
]
ENERGY_COMPANY_SUFFIXES = [
    "Energy", "Oil & Gas", "Resources", "Petroleum", "Power", "Industrial",
    "Oilfield Services", "Pipeline", "Midstream Partners", "Energy Solutions",
    "Power Systems", "Field Services", "Compression", "Utilities",
]

# Standard Salesforce Account picklist values
ACCOUNT_TYPES   = ["Prospect", "Customer", "Reseller", "Partner"]
ACCOUNT_SOURCES = ["Web", "Phone Inquiry", "Partner Referral", "Other"]
# Standard Salesforce Industry picklist values
# Note: "Manufacturing" is NOT standard — "Machinery" is the correct standard value
INDUSTRIES      = ["Energy", "Utilities", "Chemicals", "Engineering", "Construction", "Machinery", "Environmental"]

# Equipment product lines sold by our fictional industrial equipment company
PRODUCT_LINES = [
    "Industrial Generator", "Dual-Fuel Generator Set", "Gas Compression System",
    "Centrifugal Compressor", "Reciprocating Compressor", "Pump Package",
    "Power Distribution Unit", "Flare System", "Control Panel & SCADA",
    "Separator Package", "Heat Exchanger", "Switchgear Assembly",
    "Emergency Power System", "VFD Drive Package", "Wellhead Equipment Package",
]

CUSTOMER_FACILITIES = [
    "upstream field site", "midstream compression station", "refinery",
    "gas processing plant", "LNG terminal", "offshore platform",
    "produced water facility", "pipeline pump station", "power substation",
    "chemical plant", "mining operation", "distribution terminal",
]

LOCATIONS = [
    "Permian Basin", "West Texas", "Gulf of Mexico", "Eagle Ford Shale",
    "Bakken Formation", "Marcellus Shale", "Gulf Coast", "Midland Basin",
    "Delaware Basin", "Utica Shale", "DJ Basin", "Anadarko Basin",
    "Offshore Louisiana", "Southeast Texas", "Alberta Oil Sands",
]

# ─────────────────────────────────────────────────────────────────────────────
# Opportunity content templates
# ─────────────────────────────────────────────────────────────────────────────
OPP_ACTIONS = [
    "Procurement", "Fleet Replacement", "Capacity Expansion", "Upgrade",
    "New Installation", "Rental Agreement", "Service Contract", "Lease",
]

OPP_DESCRIPTION_TEMPLATES = [
    "{company} requires {qty}x {product} units for their {location} {facility}. "
    "Existing equipment is approaching end-of-life and continuous uptime is critical to production targets.",

    "{company} is expanding their {location} {facility} and needs additional {product} capacity "
    "to support increased throughput. Target commissioning is Q{quarter} {year}.",

    "Following a recent equipment failure, {company} is seeking a reliable replacement {product} "
    "at their {location} {facility}. Uptime SLA of 99.5% is contractually required.",

    "{company} submitted an RFQ for a {qty}MW {product} package to support backup power requirements "
    "at their {location} {facility}. Capital budget has been approved.",

    "Planned capital project at {company}'s {location} {facility} includes procurement of a new {product}. "
    "Engineering specifications have been finalized and vendor selection is underway.",

    "{company}'s operations team has identified the current {product} at their {location} {facility} "
    "as a throughput bottleneck. They are evaluating replacement and upgrade options for Q{quarter} {year}.",

    "Multi-site {product} standardization initiative across {company}'s {location} assets. "
    "This award represents the first of potentially {qty} units over a 3-year rollout.",
]

# ─────────────────────────────────────────────────────────────────────────────
# Case content templates
# ─────────────────────────────────────────────────────────────────────────────
CASE_SUBJECT_TEMPLATES = [
    "{product} tripping offline under high load – {location}",
    "High temperature alarm on {product} unit #{unit}",
    "Oil pressure fault on {product} – urgent response needed",
    "{product} control panel showing fault code E{code}",
    "Unusual vibration detected on {product} at {location} site",
    "Coolant leak on {product} unit #{unit} – load reduced",
    "{product} failed to start – {location} site on backup",
    "Fuel efficiency below specification on {product} post-service",
    "Spare parts request – {product} fuel injectors and seals",
    "Request for updated O&M manual – {product} (serial #{unit})",
    "{product} scheduled maintenance – technician dispatch request",
    "Emissions exceedance on {product} – regulatory deadline approaching",
    "{product} load-sharing issue – units not synchronizing",
    "Remote monitoring telemetry loss – {product} unit #{unit}",
]

CASE_DESCRIPTION_TEMPLATES = [
    "Unit #{unit} at the {company} {location} {facility} has been tripping offline when load exceeds {pct}% "
    "capacity. The fault log shows high coolant temperature alarms immediately before each shutdown. Operators "
    "performed a manual reset but the issue recurred within two hours. Customer is requesting urgent on-site "
    "technical support and a formal root-cause analysis report.",

    "The {product} installed at {company}'s {location} {facility} is displaying fault code E{code} on the main "
    "control panel. The unit has been isolated from the load bus as a precaution. The customer reports no recent "
    "parameter changes or maintenance activity that would explain the fault. Please advise on diagnostic steps "
    "and confirm parts availability.",

    "An unusual low-frequency vibration has been detected on {product} unit #{unit} at the {company} {location} "
    "site over the past week. Vibration amplitude is approximately {pct}% above baseline per SCADA trending data. "
    "The unit remains online but the site team requests an inspection at the earliest opportunity to prevent an "
    "unplanned production outage.",

    "A coolant leak has been identified on {product} unit #{unit} at {company}'s {location} {facility}. The leak "
    "appears to originate from the heat exchanger section. Customer has reduced load to 60% as a precaution and "
    "is requesting a field service technician and the required gasket and seal parts under the existing "
    "maintenance agreement.",

    "The {product} at {company}'s {location} site failed to start during a scheduled switchover test. The starter "
    "motor engages but the unit does not reach ignition speed. The backup unit is currently carrying full load. "
    "Customer requests an emergency service call and availability of a temporary rental unit while repairs are "
    "completed.",

    "{company}'s operations team reports that {product} unit #{unit} is consuming approximately {pct}% more fuel "
    "than the rated specification following the last scheduled service. They have attached the service report and "
    "fuel consumption trending data for review. Requesting a technical investigation to determine whether the "
    "issue is related to the recent service work.",

    "{company} is requesting an updated Operations & Maintenance manual and spare parts catalog for their {product} "
    "(serial #{unit}) at the {location} {facility}. The documentation currently on-site is several revisions out "
    "of date and the site manager requires the latest version ahead of an upcoming third-party HSE audit.",
]

# ─────────────────────────────────────────────────────────────────────────────
# Salesforce standard picklist values
# ─────────────────────────────────────────────────────────────────────────────
OPPORTUNITY_STAGES = [
    "Prospecting", "Qualification", "Needs Analysis", "Value Proposition",
    "Id. Decision Makers", "Perception Analysis", "Proposal/Price Quote",
    "Negotiation/Review", "Closed Won", "Closed Lost",
]
STAGE_PROBABILITIES = {
    "Prospecting": 10, "Qualification": 20, "Needs Analysis": 30,
    "Value Proposition": 50, "Id. Decision Makers": 60, "Perception Analysis": 70,
    "Proposal/Price Quote": 75, "Negotiation/Review": 90,
    "Closed Won": 100, "Closed Lost": 0,
}
OPEN_STAGE_ORDER = [
    "Prospecting", "Qualification", "Needs Analysis", "Value Proposition",
    "Id. Decision Makers", "Perception Analysis", "Proposal/Price Quote", "Negotiation/Review",
]
OPPORTUNITY_TYPES = [
    "New Customer", "Existing Customer - Upgrade",
    "Existing Customer - Replacement", "Existing Customer - Downgrade",
]
LEAD_SOURCES = [
    "Web", "Phone Inquiry", "Partner Referral", "Trade Show",
    "Internal", "Employee Referral", "External Referral", "Other",
]
CASE_STATUSES   = ["New", "Working", "Escalated", "Closed"]
CASE_ORIGINS    = ["Phone", "Email", "Web"]
CASE_PRIORITIES = ["Low", "Medium", "High"]

print(f"Configuration loaded — generating {NUM_ACCOUNTS} Accounts, "
      f"{MIN_OPPS_PER_ACCOUNT}–{MAX_OPPS_PER_ACCOUNT} Opps/Account, "
      f"{MIN_CASES_PER_ACCOUNT}–{MAX_CASES_PER_ACCOUNT} Cases/Account.")
print(f"Output directory : {OUTPUT_DIR}")
print(f"Pipeline model   : O&G services profile, update up to {PAST_OPP_UPDATE_LIMIT} overdue opps, then create future pipeline")

Configuration loaded — generating 50 Accounts, 1–5 Opps/Account, 0–8 Cases/Account.
Output directory : c:/temp
Pipeline model   : O&G services profile, update up to 200 overdue opps, then create future pipeline


## 3. Generate Synthetic Account Data

In [5]:

def _random_date(start: datetime, end: datetime) -> datetime:
    delta = end - start
    return start + timedelta(seconds=random.randint(0, int(delta.total_seconds())))


def _energy_company_name() -> str:
    return f"{random.choice(ENERGY_COMPANY_PREFIXES)} {random.choice(ENERGY_COMPANY_SUFFIXES)}"


def generate_accounts(n: int = NUM_ACCOUNTS) -> pd.DataFrame:
    records = []
    for _ in range(n):
        location = random.choice(LOCATIONS)
        facility = random.choice(CUSTOMER_FACILITIES)
        records.append({
            "Name":              _energy_company_name(),
            "Type":              random.choice(ACCOUNT_TYPES),
            "Industry":          random.choice(INDUSTRIES),
            "AccountSource":     random.choice(ACCOUNT_SOURCES),
            "Phone":             fake.phone_number(),
            "Website":           fake.url(),
            "AnnualRevenue":     round(random.uniform(5_000_000, 2_000_000_000), 2),
            "NumberOfEmployees": random.randint(25, 100_000),
            "BillingStreet":     fake.street_address(),
            "BillingCity":       fake.city(),
            # Use full state name — required when State/Country picklists are enabled
            "BillingState":      fake.state(),
            "BillingPostalCode": fake.zipcode(),
            # Use full country label, not ISO code "US" — required when Country picklists are enabled
            "BillingCountry":    "United States",
            "Description":       (
                f"Energy sector operator with primary assets in {location}. "
                f"Operations include {facility} activities requiring reliable power, "
                f"compression, and fluid handling equipment."
            ),
        })
    df = pd.DataFrame(records)
    print(f"Generated {len(df)} Account records.")
    return df


df_accounts = generate_accounts()
df_accounts.head()


Generated 50 Account records.


,Name,Type,Industry,AccountSource,Phone,Website,AnnualRevenue,NumberOfEmployees,BillingStreet,BillingCity,BillingState,BillingPostalCode,BillingCountry,Description
0,Permian Field Services,Reseller,Utilities,Phone Inquiry,+1-210-343-3218x1960,http://www.guzman.net/,2.833782e+08,13459,79402 Peterson Drives Apt. 511,Davisstad,Rhode Island,35172,United States,Energy sector operator with primary assets in ...
1,Lone Star Energy Solutions,Partner,Energy,Web,7785161849,http://www.perez.com/,1.919220e+08,30520,4752 Kelly Skyway,Jacquelineland,Rhode Island,83728,United States,Energy sector operator with primary assets in ...
2,Permian Midstream Partners,Customer,Machinery,Other,(783)227-6483,http://www.morales-jones.org/,4.447790e+08,77261,76724 John Points Suite 969,Coxberg,North Carolina,65187,United States,Energy sector operator with primary assets in ...
3,Marcellus Field Services,Partner,Chemicals,Partner Referral,801-922-6916,https://www.le.com/,3.151816e+08,44143,46270 Stanton Track Apt. 814,East Nathaniel,Hawaii,71198,United States,Energy sector operator with primary assets in ...
4,Pacific Rim Oil & Gas,Reseller,Environmental,Partner Referral,001-380-395-7015x43039,https://www.cannon.net/,1.209433e+09,5720,8963 Jennifer Locks,Samuelhaven,New Hampshire,16361,United States,Energy sector operator with primary assets in ...


## 4. Generate Synthetic Opportunity Data

In [6]:
def _opp_description(company: str, product: str, location: str, facility: str) -> str:
    template = random.choice(OPP_DESCRIPTION_TEMPLATES)
    return template.format(
        company=company,
        product=product,
        location=location,
        facility=facility,
        qty=random.randint(1, 12),
        quarter=random.randint(1, 4),
        year=random.randint(2026, 2028),
    )


# Oil & gas services conversion profile (stage-specific tendencies)
STAGE_WIN_RATE_PROFILE = {
    "Prospecting": 0.06,
    "Qualification": 0.10,
    "Needs Analysis": 0.16,
    "Value Proposition": 0.27,
    "Id. Decision Makers": 0.38,
    "Perception Analysis": 0.47,
    "Proposal/Price Quote": 0.60,
    "Negotiation/Review": 0.72,
}

OVERDUE_CLOSE_PROB_PROFILE = {
    "Prospecting": 0.30,
    "Qualification": 0.35,
    "Needs Analysis": 0.40,
    "Value Proposition": 0.46,
    "Id. Decision Makers": 0.54,
    "Perception Analysis": 0.60,
    "Proposal/Price Quote": 0.68,
    "Negotiation/Review": 0.78,
}

ADVANCE_PROB_PROFILE = {
    "Prospecting": 0.26,
    "Qualification": 0.29,
    "Needs Analysis": 0.31,
    "Value Proposition": 0.30,
    "Id. Decision Makers": 0.28,
    "Perception Analysis": 0.24,
    "Proposal/Price Quote": 0.20,
    "Negotiation/Review": 0.16,
}


def _open_stage_from_close_horizon(days_to_close: int) -> str:
    # Longer-cycle profile typical in oil & gas services: more volume in early/mid stages.
    if days_to_close <= 30:
        choices = ["Negotiation/Review", "Proposal/Price Quote", "Perception Analysis", "Id. Decision Makers"]
        weights = [0.42, 0.30, 0.18, 0.10]
    elif days_to_close <= 90:
        choices = ["Proposal/Price Quote", "Perception Analysis", "Id. Decision Makers", "Value Proposition", "Negotiation/Review"]
        weights = [0.32, 0.24, 0.18, 0.16, 0.10]
    elif days_to_close <= 180:
        choices = ["Needs Analysis", "Value Proposition", "Id. Decision Makers", "Perception Analysis", "Qualification"]
        weights = [0.28, 0.24, 0.20, 0.14, 0.14]
    else:
        choices = ["Prospecting", "Qualification", "Needs Analysis", "Value Proposition"]
        weights = [0.36, 0.30, 0.22, 0.12]

    return random.choices(choices, weights=weights, k=1)[0]


def _amount_from_stage(stage: str) -> float:
    base = round(random.uniform(80_000, 8_500_000), 2)
    stage_multiplier = {
        "Prospecting": 0.80,
        "Qualification": 0.88,
        "Needs Analysis": 0.94,
        "Value Proposition": 1.00,
        "Id. Decision Makers": 1.08,
        "Perception Analysis": 1.10,
        "Proposal/Price Quote": 1.18,
        "Negotiation/Review": 1.25,
    }.get(stage, 1.00)
    return round(base * stage_multiplier, 2)


def generate_future_pipeline_opportunities(accounts_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    now = datetime.now()

    for _, account in accounts_df.iterrows():
        num_opps = random.randint(MIN_OPPS_PER_ACCOUNT, MAX_OPPS_PER_ACCOUNT)

        for _ in range(num_opps):
            product  = random.choice(PRODUCT_LINES)
            location = random.choice(LOCATIONS)
            facility = random.choice(CUSTOMER_FACILITIES)
            action   = random.choice(OPP_ACTIONS)

            close_days = random.randint(14, CLOSE_DATE_FUTURE_DAYS)
            stage      = _open_stage_from_close_horizon(close_days)
            close_date = now + timedelta(days=close_days)

            records.append({
                "AccountId":   account["_sf_id"],
                "Name":        f"{product} – {action} ({location})",
                "StageName":   stage,
                "Amount":      _amount_from_stage(stage),
                "CloseDate":   close_date.strftime("%Y-%m-%d"),
                "Probability": STAGE_PROBABILITIES[stage],
                "Type":        random.choice(OPPORTUNITY_TYPES),
                "LeadSource":  random.choice(LEAD_SOURCES),
                "Description": _opp_description(account["Name"], product, location, facility),
            })

    df = pd.DataFrame(records)
    print(f"Generated {len(df)} future-pipeline Opportunity records across {len(accounts_df)} Accounts.")
    return df


def fetch_overdue_open_opportunities(sf: Salesforce, limit: int = PAST_OPP_UPDATE_LIMIT) -> pd.DataFrame:
    soql = f"""
        SELECT Id, Name, StageName, CloseDate, Probability, Amount
        FROM Opportunity
        WHERE IsClosed = false
          AND CloseDate < TODAY
          AND CloseDate = LAST_N_DAYS:{PIPELINE_LOOKBACK_DAYS}
        ORDER BY CloseDate ASC
        LIMIT {limit}
    """
    result = sf.query_all(soql)
    records = result.get("records", [])
    for r in records:
        r.pop("attributes", None)

    df = pd.DataFrame(records)
    print(f"Fetched {len(df)} overdue open opportunities to update.")
    return df


def _next_open_stage(current_stage: str) -> str:
    if current_stage not in OPEN_STAGE_ORDER:
        return "Qualification"

    idx = OPEN_STAGE_ORDER.index(current_stage)
    r = random.random()
    advance_prob = ADVANCE_PROB_PROFILE.get(current_stage, 0.25)

    if r < advance_prob and idx < len(OPEN_STAGE_ORDER) - 1:
        return OPEN_STAGE_ORDER[idx + 1]
    if r < (advance_prob + 0.10) and idx > 0:
        return OPEN_STAGE_ORDER[idx - 1]
    return current_stage


def build_overdue_outcome_updates(overdue_df: pd.DataFrame) -> list[dict]:
    if overdue_df.empty:
        return []

    updates = []
    today = datetime.now().date()

    for _, opp in overdue_df.iterrows():
        stage = opp.get("StageName") if opp.get("StageName") in OPEN_STAGE_ORDER else "Qualification"
        close_dt = datetime.strptime(str(opp["CloseDate"]), "%Y-%m-%d").date()
        overdue_days = max((today - close_dt).days, 1)

        # Age penalty: older overdue deals are more likely to close out and less likely to win.
        age_factor = min(0.20, overdue_days / 900)
        close_prob = min(0.90, OVERDUE_CLOSE_PROB_PROFILE.get(stage, 0.45) + age_factor)

        if random.random() <= close_prob:
            raw_win_rate = STAGE_WIN_RATE_PROFILE.get(stage, PAST_OPP_BASE_WIN_RATE)
            win_rate = max(0.08, min(0.82, raw_win_rate - (overdue_days / 1100)))
            final_stage = "Closed Won" if random.random() <= win_rate else "Closed Lost"
            next_close = close_dt + timedelta(days=random.randint(0, 21))
        else:
            # Keep it open and move organically through the pipeline.
            final_stage = _next_open_stage(stage)
            next_close = today + timedelta(days=random.randint(21, CLOSE_DATE_FUTURE_DAYS))

        updates.append({
            "Id": opp["Id"],
            "StageName": final_stage,
            "Probability": STAGE_PROBABILITIES[final_stage],
            "CloseDate": next_close.strftime("%Y-%m-%d"),
        })

    return updates


def update_opportunities(sf: Salesforce, updates: list[dict]) -> tuple[int, list[dict]]:
    success = 0
    failed = []

    for rec in updates:
        opp_id = rec["Id"]
        payload = {k: v for k, v in rec.items() if k != "Id"}
        try:
            sf.Opportunity.update(opp_id, payload)
            success += 1
        except Exception as e:
            failed.append({"record": rec, "error": str(e)})

    return success, failed


# Use placeholder IDs for local preview (real IDs come from API insert in Section 8)
_preview_accounts = df_accounts.copy()
_preview_accounts["_sf_id"] = [f"001PREVIEW{i:010d}" for i in range(len(_preview_accounts))]
df_opportunities_preview = generate_future_pipeline_opportunities(_preview_accounts)
df_opportunities_preview.head()

Generated 158 future-pipeline Opportunity records across 50 Accounts.


,AccountId,Name,StageName,Amount,CloseDate,Probability,Type,LeadSource,Description
0,001PREVIEW0000000000,Reciprocating Compressor – New Installation (S...,Qualification,3031856.96,2026-12-11,20,Existing Customer - Replacement,Web,Permian Field Services requires 5x Reciprocati...
1,001PREVIEW0000000000,Reciprocating Compressor – Service Contract (P...,Needs Analysis,2557893.21,2026-12-08,30,New Customer,External Referral,Planned capital project at Permian Field Servi...
2,001PREVIEW0000000000,Switchgear Assembly – Upgrade (Gulf Coast),Qualification,7101180.39,2026-12-18,20,Existing Customer - Replacement,Employee Referral,Permian Field Services's operations team has i...
3,001PREVIEW0000000001,Heat Exchanger – Service Contract (Gulf Coast),Id. Decision Makers,1830917.07,2026-11-13,60,Existing Customer - Downgrade,Partner Referral,Planned capital project at Lone Star Energy So...
4,001PREVIEW0000000001,Control Panel & SCADA – New Installation (Sout...,Needs Analysis,6294323.97,2026-11-08,30,Existing Customer - Replacement,Other,Lone Star Energy Solutions submitted an RFQ fo...


## 5. Generate Synthetic Case Data

In [7]:

def _case_subject(product: str, location: str) -> str:
    template = random.choice(CASE_SUBJECT_TEMPLATES)
    return template.format(
        product=product,
        location=location,
        unit=random.randint(1, 20),
        code=random.randint(100, 999),
    )


def _case_description(company: str, product: str, location: str, facility: str) -> str:
    template = random.choice(CASE_DESCRIPTION_TEMPLATES)
    return template.format(
        company=company,
        product=product,
        location=location,
        facility=facility,
        unit=random.randint(1, 20),
        code=random.randint(100, 999),
        pct=random.randint(15, 40),
    )


def generate_cases(accounts_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    case_counter = 1000

    for _, account in accounts_df.iterrows():
        num_cases = random.randint(MIN_CASES_PER_ACCOUNT, MAX_CASES_PER_ACCOUNT)
        account_created = _random_date(CREATED_DATE_START, CREATED_DATE_END)

        for _ in range(num_cases):
            product  = random.choice(PRODUCT_LINES)
            location = random.choice(LOCATIONS)
            facility = random.choice(CUSTOMER_FACILITIES)

            records.append({
                "AccountId":   account["_sf_id"],
                "Subject":     _case_subject(product, location),
                "Description": _case_description(account["Name"], product, location, facility),
                "Status":      random.choice(CASE_STATUSES),
                "Priority":    random.choice(CASE_PRIORITIES),
                "Origin":      random.choice(CASE_ORIGINS),
                # Type and Reason are omitted — their picklist values vary widely by org
                "IsEscalated": random.choice([True, False]),
            })
            case_counter += 1

    df = pd.DataFrame(records)
    print(f"Generated {len(df)} Case records across {len(accounts_df)} Accounts.")
    return df


df_cases_preview = generate_cases(_preview_accounts)
df_cases_preview.head()


Generated 180 Case records across 50 Accounts.


,AccountId,Subject,Description,Status,Priority,Origin,IsEscalated
0,001PREVIEW0000000000,Wellhead Equipment Package tripping offline un...,Permian Field Services's operations team repor...,Closed,Low,Web,True
1,001PREVIEW0000000000,Spare parts request – Power Distribution Unit ...,An unusual low-frequency vibration has been de...,Escalated,Medium,Email,False
2,001PREVIEW0000000000,Fuel efficiency below specification on Control...,Permian Field Services's operations team repor...,Working,Medium,Email,False
3,001PREVIEW0000000002,Spare parts request – Gas Compression System f...,Permian Midstream Partners's operations team r...,Working,Medium,Phone,False
4,001PREVIEW0000000002,Fuel efficiency below specification on Separat...,Permian Midstream Partners's operations team r...,Closed,High,Web,False


## 6. Salesforce API Authentication Setup

This notebook inserts records via the [**simple-salesforce**](https://github.com/simple-salesforce/simple-salesforce) REST API client.  
Two auth methods are supported — pick one below.

---

### Option A — Username + Password + Security Token *(quickest for dev/sandbox)*

1. Log in to Salesforce → **Settings → My Personal Information → Reset My Security Token**
2. Salesforce emails you the token (append it to your password, or supply it separately)
3. Set `AUTH_METHOD = "password"` and populate the variables below (or in a `.env` file)
4. Use `SF_DOMAIN = "test"` for sandboxes, `"login"` for production

---

### Option B — OAuth 2.0 Connected App + Client Credentials *(recommended for automation)*

**Step 1 — Create a Connected App in Salesforce**
1. Setup → **App Manager → New Connected App**
2. Check ✅ **Enable OAuth Settings**
3. Set **Callback URL** to `https://login.salesforce.com/services/oauth2/success`
4. Add OAuth Scopes: `api`, `refresh_token`, `offline_access`
5. Check ✅ **Enable Client Credentials Flow**
6. Save → copy the **Consumer Key** (`SF_CLIENT_ID`) and **Consumer Secret** (`SF_CLIENT_SECRET`)

**Step 2 — Assign a Run As user (Client Credentials only)**
- Setup → **Manage Connected Apps** → your app → **Edit Policies**
- Set *Client Credentials Flow* → *Run As* to a user with API + object permissions

**Step 3 — Populate credentials below (or in `.env`)**

---

> ⚠️ **Security:** Never commit credentials to source control. Use a `.env` file (already in `.gitignore`) or OS environment variables.

In [8]:
# All values load from the [Salesforce] section of .env
AUTH_METHOD = os.getenv("SF_AUTH_METHOD", "client_credentials")  # "password" | "client_credentials"

# Option A — Username / Password / Security Token
SF_USERNAME       = os.getenv("SF_USERNAME", "")
SF_PASSWORD       = os.getenv("SF_PASSWORD", "")
SF_SECURITY_TOKEN = os.getenv("SF_SECURITY_TOKEN", "")

# Option B — Connected App Client Credentials
SF_CLIENT_ID      = os.getenv("SF_CLIENT_ID", "")
SF_CLIENT_SECRET  = os.getenv("SF_CLIENT_SECRET", "")

# My Domain hostname — used as the OAuth token endpoint for client_credentials flow
# Find it at: Setup → My Domain   (e.g. orgfarm-b24e82440c-dev-ed.develop.my.salesforce.com)
SF_MY_DOMAIN = os.getenv("SF_MY_DOMAIN", "")

# "test" for sandbox, "login" for production (used only for password auth)
SF_DOMAIN = os.getenv("SF_DOMAIN", "login")

# Batch size for REST API calls (max 200 per request)
API_BATCH_SIZE = int(os.getenv("SF_API_BATCH_SIZE", "200"))

print(f"Auth method : {AUTH_METHOD}")
print(f"My Domain   : {SF_MY_DOMAIN or '(not set)'}")
print(f"Batch size  : {API_BATCH_SIZE}")
print(
    "Credentials loaded from environment."
    if SF_USERNAME or SF_CLIENT_ID
    else "⚠️  No credentials set — populate the Salesforce section of .env."
)


Auth method : client_credentials
My Domain   : orgfarm-b24e82440c-dev-ed.develop.my.salesforce.com
Batch size  : 200
Credentials loaded from environment.


## 7. Connect to Salesforce

In [9]:
import requests

def connect_to_salesforce() -> Salesforce:
    if AUTH_METHOD == "password":
        sf = Salesforce(
            username=SF_USERNAME,
            password=SF_PASSWORD,
            security_token=SF_SECURITY_TOKEN,
            domain=SF_DOMAIN,
        )
    elif AUTH_METHOD == "client_credentials":
        if not SF_MY_DOMAIN:
            raise ValueError("SF_MY_DOMAIN is not set. Add it to your .env file.")
        token_url = f"https://{SF_MY_DOMAIN}/services/oauth2/token"
        resp = requests.post(token_url, data={
            "grant_type":    "client_credentials",
            "client_id":     SF_CLIENT_ID,
            "client_secret": SF_CLIENT_SECRET,
        })
        if not resp.ok:
            print(f"Token request failed — Status: {resp.status_code}")
            print(f"URL: {token_url}")
            print(f"Response: {resp.json()}")
        resp.raise_for_status()
        token_data = resp.json()
        sf = Salesforce(
            instance_url=token_data["instance_url"],
            session_id=token_data["access_token"],
        )
    else:
        raise ValueError(f"Unknown AUTH_METHOD: {AUTH_METHOD!r}")

    print(f"Connected to Salesforce org: {sf.sf_instance}")
    return sf


sf = connect_to_salesforce()


Connected to Salesforce org: orgfarm-b24e82440c-dev-ed.develop.my.salesforce.com


## 8. Simulate Oil & Gas Pipeline Lifecycle + Insert New Records

This section now mimics a realistic oil and gas services pipeline in two phases:

1. Finds **existing overdue open opportunities** and applies organic lifecycle behavior by stage:
   - some are closed as **Won/Lost** using stage-based conversion rates,
   - others are **re-forecasted** (close date pushed) with realistic stage progression/regression.
2. Inserts **new future pipeline opportunities** with horizon-based stage distribution, then inserts Cases for newly created Accounts.

Failed records are logged and skipped without stopping the batch.

In [ ]:
import re

def _extract_account_id_from_error(error_text: str) -> str | None:
    # Parse an Account Id from duplicate-rule error payloads when Salesforce returns a match URL.
    match = re.search(r"/sobjects/Account/(001[a-zA-Z0-9]{12,15})", error_text)
    return match.group(1) if match else None


def insert_records(sf_object, records: list[dict], allow_duplicate_save: bool = False) -> tuple[list[str], list[dict]]:
    """
    Insert records in batches using the Salesforce REST API.
    Returns (list_of_inserted_ids, list_of_failed_records).
    """
    sf_type = getattr(sf, sf_object)
    inserted_ids = []
    failed = []

    for i in range(0, len(records), API_BATCH_SIZE):
        batch = records[i : i + API_BATCH_SIZE]
        for rec in batch:
            try:
                if allow_duplicate_save:
                    try:
                        result = sf_type.create(
                            rec,
                            headers={"Sforce-Duplicate-Rule-Header": "allowSave=true"},
                        )
                    except TypeError:
                        # Some simple-salesforce versions may not accept the headers kwarg.
                        result = sf_type.create(rec)
                else:
                    result = sf_type.create(rec)

                inserted_ids.append(result["id"])
            except Exception as e:
                error_text = str(e)

                # If duplicate rules block Account insert, try to reuse the matched Account Id.
                if sf_object == "Account" and "DUPLICATES_DETECTED" in error_text:
                    existing_id = _extract_account_id_from_error(error_text)
                    if existing_id:
                        inserted_ids.append(existing_id)
                        continue

                failed.append({"record": rec, "error": error_text})

    return inserted_ids, failed


# ── Step 1: Update overdue open opportunities to closed outcomes ───────────────
try:
    past_due_df = fetch_overdue_open_opportunities(sf, limit=PAST_OPP_UPDATE_LIMIT)
    past_due_updates = build_overdue_outcome_updates(past_due_df)
    updated_opp_count, updated_opp_failures = update_opportunities(sf, past_due_updates)
except Exception as e:
    print(f"Warning: overdue opportunity update step skipped due to API error: {e}")
    past_due_df = pd.DataFrame()
    past_due_updates = []
    updated_opp_count = 0
    updated_opp_failures = [{"record": {}, "error": str(e)}]

print(f"Updated overdue Opportunities -> ✅ Updated: {updated_opp_count}   ❌ Failed: {len(updated_opp_failures)}")

# ── Step 2: Insert Accounts ───────────────────────────────────────────────────
account_records = df_accounts.to_dict(orient="records")
print(f"\nInserting {len(account_records)} Accounts...")
account_ids, account_failures = insert_records("Account", account_records, allow_duplicate_save=True)
print(f"  ✅ Inserted/Reused: {len(account_ids)}   ❌ Failed: {len(account_failures)}")

# Attach real SF IDs back to the dataframe
df_accounts_inserted = df_accounts.copy()
df_accounts_inserted["_sf_id"] = account_ids + [None] * len(account_failures)
df_accounts_inserted = df_accounts_inserted.dropna(subset=["_sf_id"])

# ── Step 3: Generate future pipeline Opportunities & Cases for new Accounts ───
df_opportunities = generate_future_pipeline_opportunities(df_accounts_inserted)
df_cases         = generate_cases(df_accounts_inserted)

# ── Step 4: Insert Opportunities ─────────────────────────────────────────────
opp_records = df_opportunities.to_dict(orient="records")
print(f"\nInserting {len(opp_records)} new future-pipeline Opportunities...")
opp_ids, opp_failures = insert_records("Opportunity", opp_records)
print(f"  ✅ Inserted: {len(opp_ids)}   ❌ Failed: {len(opp_failures)}")

# ── Step 5: Insert Cases ──────────────────────────────────────────────────────
case_records = df_cases.to_dict(orient="records")
print(f"\nInserting {len(case_records)} Cases...")
case_ids, case_failures = insert_records("Case", case_records)
print(f"  ✅ Inserted: {len(case_ids)}   ❌ Failed: {len(case_failures)}")

if updated_opp_failures or opp_failures or case_failures or account_failures:
    print("\n── Failed Records ──")
    for f in updated_opp_failures + account_failures + opp_failures + case_failures:
        rec_preview = list(f["record"].items())[:3] if isinstance(f.get("record"), dict) else f.get("record")
        print(f"  {f['error']}  |  {rec_preview}")

Updated overdue Opportunities -> ✅ Updated: 0   ❌ Failed: 1

Inserting 50 Accounts...


## 9. Export Data Locally (CSV / JSON / Excel)

Run this section **independently** to save the generated data locally without inserting into Salesforce — useful for review or Salesforce Data Loader imports.

In [ ]:
import pathlib

pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Use preview DataFrames if API insert hasn't been run yet
_export_accounts = df_accounts_inserted if "df_accounts_inserted" in dir() else df_accounts
_export_opps     = df_opportunities      if "df_opportunities"      in dir() else df_opportunities_preview
_export_cases    = df_cases              if "df_cases"               in dir() else df_cases_preview

# ── CSV (Salesforce Data Loader-compatible) ───────────────────────────────────
_export_accounts.drop(columns=["_sf_id"], errors="ignore").to_csv(f"{OUTPUT_DIR}/sf_accounts.csv",     index=False)
_export_opps    .drop(columns=["_sf_id"], errors="ignore").to_csv(f"{OUTPUT_DIR}/sf_opportunities.csv", index=False)
_export_cases   .drop(columns=["_sf_id"], errors="ignore").to_csv(f"{OUTPUT_DIR}/sf_cases.csv",         index=False)
print(f"CSV files written to {OUTPUT_DIR}/")

# ── JSON ─────────────────────────────────────────────────────────────────────
_export_accounts.drop(columns=["_sf_id"], errors="ignore").to_json(f"{OUTPUT_DIR}/sf_accounts.json",     orient="records", indent=2)
_export_opps    .drop(columns=["_sf_id"], errors="ignore").to_json(f"{OUTPUT_DIR}/sf_opportunities.json", orient="records", indent=2)
_export_cases   .drop(columns=["_sf_id"], errors="ignore").to_json(f"{OUTPUT_DIR}/sf_cases.json",         orient="records", indent=2)
print(f"JSON files written to {OUTPUT_DIR}/")

# ── Excel (multi-sheet workbook) ──────────────────────────────────────────────
excel_path = f"{OUTPUT_DIR}/sf_synthetic_data.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    _export_accounts.drop(columns=["_sf_id"], errors="ignore").to_excel(writer, sheet_name="Accounts",     index=False)
    _export_opps    .drop(columns=["_sf_id"], errors="ignore").to_excel(writer, sheet_name="Opportunities", index=False)
    _export_cases   .drop(columns=["_sf_id"], errors="ignore").to_excel(writer, sheet_name="Cases",         index=False)
print(f"Excel workbook written to {excel_path}")

print("\nExport complete.")
print(f"  Accounts:      {len(_export_accounts)} rows")
print(f"  Opportunities: {len(_export_opps)} rows")
print(f"  Cases:         {len(_export_cases)} rows")

In [14]:
summary = {
    "updated_opp_count": updated_opp_count if 'updated_opp_count' in globals() else None,
    "updated_opp_failures": len(updated_opp_failures) if 'updated_opp_failures' in globals() else None,
    "inserted_accounts": len(account_ids) if 'account_ids' in globals() else None,
    "account_failures": len(account_failures) if 'account_failures' in globals() else None,
    "inserted_opportunities": len(opp_ids) if 'opp_ids' in globals() else None,
    "opportunity_failures": len(opp_failures) if 'opp_failures' in globals() else None,
    "inserted_cases": len(case_ids) if 'case_ids' in globals() else None,
    "case_failures": len(case_failures) if 'case_failures' in globals() else None,
}

print(summary)

{'updated_opp_count': 38, 'updated_opp_failures': 0, 'inserted_accounts': 0, 'account_failures': 50, 'inserted_opportunities': 0, 'opportunity_failures': 0, 'inserted_cases': 0, 'case_failures': 0}


In [12]:
print("Sample account failure:")
if 'account_failures' in globals() and account_failures:
    print(account_failures[0]["error"])
else:
    print("No account failures found.")

Sample account failure:
Malformed request https://orgfarm-b24e82440c-dev-ed.develop.my.salesforce.com/services/data/v59.0/sobjects/Account/. Response content: [{'duplicateResult': {'allowSave': True, 'duplicateRule': 'Standard_Account_Duplicate_Rule', 'duplicateRuleEntityType': 'Account', 'errorMessage': 'Use one of these records?', 'matchResults': [{'entityType': 'Account', 'errors': [], 'matchEngine': 'FuzzyMatchEngine', 'matchRecords': [{'additionalInformation': [], 'fieldDiffs': [], 'matchConfidence': 100.0, 'record': {'attributes': {'type': 'Account', 'url': '/services/data/v59.0/sobjects/Account/001gL00000levCcQAI'}, 'Id': '001gL00000levCcQAI'}}], 'rule': 'Standard_Account_Match_Rule_v1_0', 'size': 1, 'success': True}]}, 'errorCode': 'DUPLICATES_DETECTED', 'message': 'Use one of these records?'}]
